# Librerías

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import PROCESSED_PATH

np.random.seed(42)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

# Carga de Datos

In [ ]:
df_users    = pd.read_csv(PROCESSED_PATH / "users.csv",    low_memory=False)
df_events   = pd.read_csv(PROCESSED_PATH / "events.csv",   low_memory=False)
df_products = pd.read_csv(PROCESSED_PATH / "products.csv", low_memory=False)

df_events["timestamp"] = pd.to_datetime(df_events["timestamp"])

print(f"users:    {df_users.shape}")
print(f"events:   {df_events.shape}")
print(f"products: {df_products.shape}")

---
# Perfil General

## Tipos y Nulos

In [ ]:
# Nulos por tabla
for nombre, df in [("users", df_users), ("events", df_events), ("products", df_products)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f"=== {nombre} ===")
    if len(nulos):
        print(nulos.to_string())
    else:
        print("  Sin nulos")
    print()

## Duplicados y cobertura

In [ ]:
print(f"Duplicados users:    {df_users.duplicated().sum()}")
print(f"Duplicados events:   {df_events.duplicated().sum()}")
print(f"Duplicados products: {df_products.duplicated().sum()}")
print()
print(f"id_user unicos en users:  {df_users['id_user'].nunique():,}")
print(f"id_user unicos en events: {df_events['id_user'].nunique():,}")
print(f"Usuarios en users sin eventos: {df_users['id_user'].nunique() - df_events['id_user'].nunique():,}")

---
# Variable Objetivo — Clicks

## Distribución global

In [ ]:
conteo = df_events["event_type"].value_counts()
ratio  = df_events["event_type"].value_counts(normalize=True).mul(100).round(1)

resumen = pd.DataFrame({"n": conteo, "%": ratio})
print(resumen)
print()
print(f"Ratio click: {ratio.get('click', 0):.1f}%")

## Evolución temporal

In [ ]:
df_clicks = df_events[df_events["event_type"] == "click"].copy()
df_opens  = df_events[df_events["event_type"] == "open"].copy()

clicks_sem = df_clicks.set_index("timestamp").resample("W")["id_event"].count().rename("clicks")
opens_sem  = df_opens.set_index("timestamp").resample("W")["id_event"].count().rename("opens")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].bar(clicks_sem.index, clicks_sem.values, width=5, color="steelblue")
axes[0].set_title("Clicks por semana")
axes[0].set_ylabel("N clicks")

axes[1].bar(opens_sem.index, opens_sem.values, width=5, color="coral")
axes[1].set_title("Aperturas por semana")
axes[1].set_ylabel("N opens")

plt.tight_layout()
plt.show()

## Clicks por sector y CTR

In [ ]:
df_ev_prod = df_events.merge(
    df_products[["id_product", "sector", "product_new", "cpl"]],
    on="id_product", how="left"
)

# Clicks absolutos por sector
clicks_sector = (
    df_ev_prod[df_ev_prod["event_type"] == "click"]
    .groupby("sector")["id_event"].count()
    .sort_values()
)

# CTR por sector
ctr_sector = (
    df_ev_prod.groupby("sector")["event_type"]
    .apply(lambda s: (s == "click").mean() * 100)
    .sort_values()
    .round(1)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clicks_sector.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Clicks por sector (absolutos)")
axes[0].set_xlabel("N clicks")

ctr_sector.plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("CTR por sector (%)")
axes[1].set_xlabel("CTR (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Clicks por product_new
clicks_prod = (
    df_ev_prod[df_ev_prod["event_type"] == "click"]
    .groupby("product_new")["id_event"].count()
    .sort_values(ascending=False)
)
print("Clicks por product_new:")
display(clicks_prod.to_frame())

---
# Análisis de Usuarios

## Variables demográficas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Edad
axes[0].hist(df_users["age"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribución de edad")
axes[0].set_xlabel("Edad")

# Género
df_users["gender"].value_counts().plot(kind="bar", ax=axes[1], color=["steelblue", "coral"])
axes[1].set_title("Género")
axes[1].tick_params(axis="x", rotation=0)

# Estado civil
df_users["civil_status"].value_counts().plot(kind="bar", ax=axes[2], color="steelblue")
axes[2].set_title("Estado civil")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_users["labor_status"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Estado laboral")
axes[0].tick_params(axis="x", rotation=20)

df_users["size_hogar"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Tamaño del hogar")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## Perfil click vs no-click

In [ ]:
# Marcar usuarios que han hecho algún click
usuarios_con_click = df_events[df_events["event_type"] == "click"]["id_user"].unique()
df_users["hizo_click"] = df_users["id_user"].isin(usuarios_con_click)

print(f"Usuarios con al menos 1 click: {df_users['hizo_click'].sum():,}  ({df_users['hizo_click'].mean()*100:.1f}%)")
print(f"Usuarios sin ningun click:     {(~df_users['hizo_click']).sum():,} ({(~df_users['hizo_click']).mean()*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Edad por grupo
for val, label, color in [(False, "Sin click", "coral"), (True, "Con click", "steelblue")]:
    df_users[df_users["hizo_click"] == val]["age"].dropna().plot(
        kind="hist", bins=25, alpha=0.6, ax=axes[0], label=label, color=color
    )
axes[0].set_title("Edad por grupo click")
axes[0].set_xlabel("Edad")
axes[0].legend()

# % tiene coche por grupo
coche_pct = df_users.groupby("hizo_click")["tiene_coche"].mean().mul(100).round(1)
coche_pct.plot(kind="bar", ax=axes[1], color=["coral", "steelblue"])
axes[1].set_title("% tiene coche por grupo click")
axes[1].set_ylabel("%")
axes[1].set_xticklabels(["Sin click", "Con click"], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Labor status por grupo click
labor_pct = (
    df_users.groupby(["hizo_click", "labor_status"])
    .size()
    .unstack(fill_value=0)
    .apply(lambda r: r / r.sum() * 100, axis=1)
    .round(1)
)
labor_pct.index = ["Sin click", "Con click"]
print("Labor status (%) por grupo click:")
display(labor_pct)

---
# Análisis de Productos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_products["sector"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Productos por sector")
axes[0].set_xlabel("N productos")

df_products["cpl"].dropna().plot(kind="hist", bins=20, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Distribución CPL")
axes[1].set_xlabel("CPL (euros)")

plt.tight_layout()
plt.show()

print("Estadisticas CPL:")
print(df_products["cpl"].describe().round(2))

---
# Comportamiento por Usuario

In [ ]:
eventos_por_usuario = df_events.groupby("id_user")["id_event"].count().rename("n_eventos")

print("Eventos por usuario:")
print(eventos_por_usuario.describe().round(2))
print()
print(f"Usuarios con 1 evento:    {(eventos_por_usuario == 1).sum():,} ({(eventos_por_usuario == 1).mean()*100:.1f}%)")
print(f"Usuarios con > 1 evento:  {(eventos_por_usuario > 1).sum():,} ({(eventos_por_usuario > 1).mean()*100:.1f}%)")
print(f"Usuarios con > 5 eventos: {(eventos_por_usuario > 5).sum():,}")

In [ ]:
# Sparsidad de la matriz usuario-producto
n_usuarios      = df_events["id_user"].nunique()
n_productos     = df_events["id_product"].nunique()
n_interacciones = len(df_events)
sparsidad       = 1 - n_interacciones / (n_usuarios * n_productos)

print(f"Usuarios unicos:       {n_usuarios:,}")
print(f"Productos unicos:      {n_productos:,}")
print(f"Interacciones totales: {n_interacciones:,}")
print(f"Sparsidad:             {sparsidad*100:.2f}%")

---
# Hallazgos — Resumen para el Pipeline

| # | Hallazgo | Impacto en el pipeline |
|---|----------|------------------------|
| 1 | **Ratio click 23,8%** tras muestreo balanceado | Modelo de propensión viable sin class weights |
| 2 | **~5.800 usuarios con click** de ~35.000 totales | Señal comportamental suficiente para segmentación |
| 3 | El **CTR varía mucho por sector** | El sector aporta señal; se incorpora como variable del modelo de propensión |
| 4 | **Sparsidad 98,1%** en matriz user-product | Collaborative filtering limitado; pesar más el contenido del producto |
| 5 | **~1 año de datos** con granularidad semanal | Permite explorar la previsión temporal (M4) |
| 6 | **Perfil demográfico diferenciable** entre grupos | Útil para segmentación y arranque en frío (poco poder predictivo en propensión, ver M2) |